# Bobby Carrot — PPO + ICM Training (T4 GPU)

Train step-by-step on **normal levels 1–7**, test generalization on **levels 8–10**.

This notebook splits the curriculum into three explicit phases so you can monitor success before proceeding:
- **Phase 1**: Levels 1-3 (basic navigation & carrot collection)
- **Phase 2**: Levels 1-5 (adds crumble tiles)
- **Phase 3**: Levels 1-7 (full training set)

Each phase ends with an **automatic evaluation** so you can verify improvement.

## 1. Setup — Clone & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/Charan20510/Mini_Project_Game.git /content/Mini_Project_Game
%cd /content/Mini_Project_Game

In [ ]:
%pip install torch numpy pandas matplotlib

## 2. Phase 1 — Train on Levels 1 to 3
We start with the simplest levels. The agent trains for 300k timesteps with:
- **Entropy 0.03**: Encourages exploration without being so high the agent can't commit to good policies
- **Rollout 512**: Longer rollouts capture more complete episodes per level for better gradients
- **Round-robin level cycling**: Ensures equal exposure to all 3 levels (prevents level 1 memorization)

In [ ]:
from pathlib import Path
from Bobby_Carrot.rl_models.config import PPOConfig, TrainingConfig, ICMConfig, LevelConfig
from Bobby_Carrot.rl_models.ppo import train_ppo

level_config_1 = LevelConfig(
    train_levels=[('normal', 1), ('normal', 2), ('normal', 3)],
    test_levels=[],
)
train_config_1 = TrainingConfig(
    total_timesteps=300_000,
    device='auto',
    checkpoint_dir=Path('checkpoints_phase1'),
    log_dir=Path('logs_phase1'),
    curriculum=False,
    observation_mode='full',
    max_steps_per_episode=300,
)
ppo_config_1 = PPOConfig(
    lr=2.5e-4,
    entropy_coeff=0.03,
    rollout_length=512,
    n_epochs=4,
    minibatch_size=128,
    cnn_channels=[32, 64, 64, 64],
)
icm_config_1 = ICMConfig(
    enabled=True,
    intrinsic_reward_scale=0.02,
)

print("Starting Phase 1 Training (Levels 1-3)...")
agent_1 = train_ppo(ppo_config_1, train_config_1, level_config_1, icm_config_1)

### Phase 1 — Evaluation
Check if the agent has learned to play **each level individually** (not just memorizing level 1).

In [ ]:
from Bobby_Carrot.rl_models.evaluate import evaluate_agent
import os

# Use best model if it exists, otherwise use final
ckpt_1 = 'checkpoints_phase1/ppo/ppo_best.pt'
if not os.path.exists(ckpt_1):
    ckpt_1 = 'checkpoints_phase1/ppo/ppo_final.pt'

print(f"Evaluating checkpoint: {ckpt_1}")
print("="*60)
results_1 = evaluate_agent(
    algo='ppo',
    checkpoint_path=ckpt_1,
    levels=[('normal', 1), ('normal', 2), ('normal', 3)],
    episodes_per_level=20,
    max_steps=300,
    observation_mode='full',
    device_str='auto',
    render=False,
)

agg = results_1['aggregate']
print(f"\n{'='*60}")
print(f"  Phase 1 AGGREGATE SUCCESS: {agg['success_rate']:.1%}")
print(f"  Per-level breakdown:")
for level_key, metrics in results_1['per_level'].items():
    print(f"    {level_key}: {metrics['success_rate']:.0%} success, avg_reward={metrics['avg_reward']:.1f}")
phase1_ok = agg['success_rate'] >= 0.30
print(f"  Ready for Phase 2? {'✅ YES' if phase1_ok else '❌ NO (consider retraining with more steps)'}")
print(f"{'='*60}")

## 3. Phase 2 — Train on Levels 1 to 5
Once the agent is confident on 1-3, we introduce crumble tiles in levels 4 & 5.
We load the Phase 1 weights and fine-tune with a lower learning rate.

In [ ]:
import os

level_config_2 = LevelConfig(
    train_levels=[('normal', i) for i in range(1, 6)],
    test_levels=[],
)
train_config_2 = TrainingConfig(
    total_timesteps=300_000,
    device='auto',
    checkpoint_dir='checkpoints_phase2',
    log_dir='logs_phase2',
    curriculum=False,
    observation_mode='full',
    max_steps_per_episode=300,
)
ppo_config_2 = PPOConfig(
    lr=1e-4,
    entropy_coeff=0.02,
    rollout_length=512,
    n_epochs=4,
    minibatch_size=128,
    cnn_channels=[32, 64, 64, 64],
)
icm_config_2 = ICMConfig(enabled=True, intrinsic_reward_scale=0.02)

# Find best Phase 1 checkpoint
resume_1 = 'checkpoints_phase1/ppo/ppo_best.pt'
if not os.path.exists(resume_1):
    resume_1 = 'checkpoints_phase1/ppo/ppo_final.pt'

print(f"Loading agent from Phase 1: {resume_1}")
print("Starting Phase 2 Training (Levels 1-5)...")
agent_2 = train_ppo(
    ppo_config_2, train_config_2, level_config_2, icm_config_2,
    resume_path=resume_1,
)

from pathlib import Path
### Phase 2 — Evaluation
Check performance on all 5 training levels.

In [ ]:
ckpt_2 = 'checkpoints_phase2/ppo/ppo_best.pt'
if not os.path.exists(ckpt_2):
    ckpt_2 = 'checkpoints_phase2/ppo/ppo_final.pt'

print(f"Evaluating checkpoint: {ckpt_2}")
print("="*60)
results_2 = evaluate_agent(
    algo='ppo',
    checkpoint_path=ckpt_2,
    levels=[('normal', i) for i in range(1, 6)],
    episodes_per_level=20,
    max_steps=300,
    observation_mode='full',
    device_str='auto',
    render=False,
)

agg = results_2['aggregate']
print(f"\n{'='*60}")
print(f"  Phase 2 AGGREGATE SUCCESS: {agg['success_rate']:.1%}")
print(f"  Per-level breakdown:")
for level_key, metrics in results_2['per_level'].items():
    print(f"    {level_key}: {metrics['success_rate']:.0%} success, avg_reward={metrics['avg_reward']:.1f}")
phase2_ok = agg['success_rate'] >= 0.25
print(f"  Ready for Phase 3? {'✅ YES' if phase2_ok else '❌ NO (consider retraining with more steps)'}")
print(f"{'='*60}")

## 4. Phase 3 — Train on Levels 1 to 7
Adding the last couple of training levels to ensure full generalization before the unseen test.

In [ ]:
level_config_3 = LevelConfig(
    train_levels=[('normal', i) for i in range(1, 8)],
    test_levels=[('normal', 8), ('normal', 9), ('normal', 10)],
)
train_config_3 = TrainingConfig(
    total_timesteps=400_000,
    device='auto',
    checkpoint_dir='checkpoints_phase3',
    log_dir='logs_phase3',
    curriculum=False,
    observation_mode='full',
    max_steps_per_episode=300,
)
ppo_config_3 = PPOConfig(
    lr=5e-5,
    entropy_coeff=0.015,
    rollout_length=512,
    n_epochs=4,
    minibatch_size=128,
    cnn_channels=[32, 64, 64, 64],
)

resume_2 = 'checkpoints_phase2/ppo/ppo_best.pt'
if not os.path.exists(resume_2):
    resume_2 = 'checkpoints_phase2/ppo/ppo_final.pt'

print(f"Loading agent from Phase 2: {resume_2}")
print("Starting Phase 3 Training (Levels 1-7)...")
agent_3 = train_ppo(
    ppo_config_3, train_config_3, level_config_3,
    icm_config=ICMConfig(enabled=True, intrinsic_reward_scale=0.01),
    resume_path=resume_2,
)

from pathlib import Path
### Phase 3 — Evaluation on Training Levels

In [ ]:
ckpt_3 = 'checkpoints_phase3/ppo/ppo_best.pt'
if not os.path.exists(ckpt_3):
    ckpt_3 = 'checkpoints_phase3/ppo/ppo_final.pt'

print(f"Evaluating checkpoint: {ckpt_3}")
print("="*60)
results_3_train = evaluate_agent(
    algo='ppo',
    checkpoint_path=ckpt_3,
    levels=[('normal', i) for i in range(1, 8)],
    episodes_per_level=20,
    max_steps=300,
    observation_mode='full',
    device_str='auto',
    render=False,
)

agg = results_3_train['aggregate']
print(f"\n{'='*60}")
print(f"  Phase 3 TRAIN LEVELS AGGREGATE: {agg['success_rate']:.1%}")
for level_key, metrics in results_3_train['per_level'].items():
    print(f"    {level_key}: {metrics['success_rate']:.0%} success")
print(f"{'='*60}")

## 5. Final Evaluation — Test Levels 8-10 (Unseen)
The ultimate test: can the agent generalize to levels it has **never seen**?

In [ ]:
print("\n" + "🎯"*30)
print("  FINAL TEST: Unseen Levels 8-10")
print("🎯"*30 + "\n")

results_test = evaluate_agent(
    algo='ppo',
    checkpoint_path=ckpt_3,
    levels=[('normal', 8), ('normal', 9), ('normal', 10)],
    episodes_per_level=20,
    max_steps=300,
    observation_mode='full',
    device_str='auto',
    render=False,
)

agg = results_test['aggregate']
target_met = agg['success_rate'] >= 0.40
print(f"\n{'='*60}")
print(f"  AGGREGATE SUCCESS: {agg['success_rate']:.1%}")
print(f"  TARGET (>40%):     {'✅ MET' if target_met else '❌ NOT MET'}")
print(f"  Per-level breakdown:")
for level_key, metrics in results_test['per_level'].items():
    print(f"    {level_key}: {metrics['success_rate']:.0%} success, avg_reward={metrics['avg_reward']:.1f}")
print(f"{'='*60}")

## 6. Save Final Weights to Drive

In [ ]:
import shutil
import os

drive_dest = '/content/drive/MyDrive/Bobby_Carrot_RL'
os.makedirs(drive_dest, exist_ok=True)

for phase in [1, 2, 3]:
    for name in ['ppo_best.pt', 'ppo_final.pt']:
        src = f'checkpoints_phase{phase}/ppo/{name}'
        if os.path.exists(src):
            dest_name = f'ppo_phase{phase}_{name.replace("ppo_", "")}'
            shutil.copy(src, os.path.join(drive_dest, dest_name))
            print(f"Copied {src} → {dest_name}")

print(f"\nAll saved to {drive_dest}")
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print('Drive flushed.')
except:
    pass
